### Taxonomic annotation of UHVDB

In [1]:
### load packages
import polars as pl

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [ ]:
%%bash
# ### Create DIAMOND db from ICTV genomes
# singularity run -B /gscratch https://community-cr-prod.seqera.io/docker/registry/v2/blobs/sha256/43/43b2261bacec58cde7252fcf3214a4aca430ccb271fb77a3a6529465c9cd4cb3/data
# # predict genes from FNA
# pyrodigal-gv \
#     -i /gscratch/stf/carsonjm/uhvdb-proteinsimilarity-r2025_09/tmp/vmrtofasta/VMR_MSL40.v2.20251013/VMR_MSL40.v2.20251013.fna \
#     -a VMR_MSL40.v2.20251013.pyrodigalgv.faa \
#     --jobs 32

# # create DIAMOND database
# diamond \
#     makedb \
#     --threads 32 \
#     --in VMR_MSL40.v2.20251013.pyrodigalgv.faa \
#     -d VMR_MSL40.v2.20251013

# # gzip FAA file
# pigz VMR_MSL40.v2.20251013.pyrodigalgv.faa

In [ ]:
%%bash
# ### Run UHVDB/proteinsimilarity workflow
# nextflow run UHVDB/proteinsimilarity \
#     -c /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/assets/configs/conf/uw_hyak.config \
#     -r main \
#     -latest \
#     --hyak_queue 'ckpt' \
#     --hyak_partition 'pedslabs' \
#     --query_fna /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_votu_reps.fna \
#     --vmr_dmnd /gscratch/stf/carsonjm/uhvdb-proteinsimilarity-r2025_09/tmp/vmrtofasta/VMR_MSL40.v2.20251013/VMR_MSL40.v2.20251013.dmnd \
#     --chunk_size 10000 \
#     --diamond_args "--masking none -k 10000 -e 1e-3 --very-sensitive" \
#     --min_score 0 \
#     --remove_tmp false \
#     --output /gscratch/stf/carsonjm/uhvdb-proteinsimilarity-r2025_09/uhvdb_r2025_10_v_VMR_MSL40.v2.20251013.tsv

In [ ]:
# ### Read data and save to parquet
# (
#     pl.read_csv('/gscratch/stf/carsonjm/uhvdb-proteinsimilarity-r2025_09/uhvdb_r2025_10_v_VMR_MSL40.v2.20251013.tsv', separator='\t', has_header=False)
#         .write_parquet('uhvdb_r2025_10_v_VMR_MSL40.v2.20251013.parquet')
# )

In [ ]:
### Count number of queries with any hits to ICTV
df = pl.read_parquet('uhvdb_r2025_10_v_VMR_MSL40.v2.20251013.parquet')

print("Number of UHVDB genomes with a hit to ICTV:",
    df.unique(pl.col('column_1')).height
)

Number of UHVDB genomes with a hit to ICTV: 217304


In [ ]:
import polars as pl

### Count number of queries with family, subfamily, genus, subgenus, species level hits
print("Number of UHVDB genomes with family hit to ICTV:",
    df.filter(pl.col('column_3') >= 5.5).unique('column_1').height
)

print("Number of UHVDB genomes with subfamily hit to ICTV:",
    df.filter(pl.col('column_3') >= 32).unique('column_1').height
)

print("Number of UHVDB genomes with genus hit to ICTV:",
    df.filter(pl.col('column_3') >= 65).unique('column_1').height
)

print("Number of UHVDB genomes with subgenus hit to ICTV:",
    df.filter(pl.col('column_3') >= 80).unique('column_1').height
)

hq_votus.group_by('family_name').len().sort('len')

Number of UHVDB genomes with family hit to ICTV: 131814
Number of UHVDB genomes with subfamily hit to ICTV: 22767
Number of UHVDB genomes with genus hit to ICTV: 7485
Number of UHVDB genomes with subgenus hit to ICTV: 3076


In [12]:
import polars as pl

# analyze UHGV phist annotations
hq_votus = (
    pl.read_csv('votus_metadata_extended.tsv',
        null_values='NULL', separator='\t', ignore_errors=True,
        columns=['uhgv_genome', 'checkv_completeness', 'viral_confidence', 'ictv_taxonomy', 'ictv_taxonomy_method', 'ictv_taxonomy_evidence', 'family_name']
    )
        .filter(
            (pl.col('checkv_completeness') >= 90) &
            (pl.col('viral_confidence') == 'Confident')
        )
)
hq_votus.group_by('family_name').len().sort('len')

family_name,len
str,u32
"""Caulimoviridae""",1
"""Mesyanzhinovviridae""",1
"""Hytrosaviridae""",1
"""Pachyviridae""",1
"""Tombusviridae""",2
…,…
"""Autographiviridae""",454
"""Salasmaviridae""",486
"""Anelloviridae""",864


In [13]:
# analyze UHGV phist annotations
votus = (
    pl.read_csv('votus_metadata_extended.tsv',
        null_values='NULL', separator='\t', ignore_errors=True,
        columns=['uhgv_genome', 'checkv_completeness', 'viral_confidence', 'ictv_taxonomy', 'ictv_taxonomy_method', 'ictv_taxonomy_evidence', 'family_name']
    )
    .filter(
        (pl.col('checkv_completeness') < 90) &
        (pl.col('viral_confidence') == 'Confident')
    )
)
votus.group_by('family_name').len().sort('len')

family_name,len
str,u32
"""Partitiviridae""",1
"""Poxviridae""",1
"""Hytrosaviridae""",1
"""Narnaviridae""",1
"""Vertoviridae""",1
…,…
"""Quimbyviridae""",1015
"""Epsilon-crassviridae""",1143
"""Microviridae""",1439
